# 🗑️ Plastic Object Detection 🤖
Please check `READ.ME` file for prerequisite installations before continuing

## 🌱 Preparing the Environment

In [ ]:
# Run GPU acceleration if available
import torch

print("ROCm / CUDA Available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU Device Name:", torch.cuda.get_device_name(0))


In [ ]:
# Import required libraries

import os
from ultralytics import YOLO
from pathlib import Path

## 🗂️ Loading the Dataset
The dataset includes `189` images, annotated in `YOLOv11` format.
  
The `7` types of classification include:
1. Bottle Cap
2. Cone Cap
3. Leaf
4. Netting
5. Plastic Bag
6. Plastic Strand
7. Sponge
  
<font color='#FFC067'>Pre-processing </font> was applied to each image <font color='#FFC067'>for consistency</font>:  
1. Auto-orientation of pixel data (with EXIF-orientation stripping)  
2. Resized to `512x512` (Stretch)

### 💾 Part 1: Importing from Roboflow
Replace the `ROBOFLOW_API_KEY` with your own in `.env`

In [ ]:
from roboflow import Roboflow
from dotenv import load_dotenv

load_dotenv()

rf = Roboflow(api_key=os.getenv("ROBOFLOW_API_KEY"))
project = rf.workspace("brittany-nguyen").project("trash-tank-yolov11")
version = project.version(4)
dataset = version.download("yolov11")

In [ ]:
from pathlib import Path
import os

# Set dataset path
dataset_path = dataset.location

# Verify dataset path
print(f"Dataset path is set to: {dataset_path}")
print(f"Files in {os.path.basename(dataset_path)}: {os.listdir(dataset_path)}")

# Set train and val paths
train_path = os.path.join(dataset_path, 'train/')
val_path = os.path.join(dataset_path, 'val/')

# 🛠️ Augmentating Image Data

In [ ]:
# Database connection libraries

import yaml
import pyodbc

# Load YAML data into SQL Server database
with open('data.yaml', 'r') as file:
    data = yaml.safe_load(file)

In [ ]:
# Set database connection parameters


# Set datapath

## 🚀 Initializing TensorBoard
TensorBoard is a <font color='#FFC067'>visualization tool</font> for real-time insights of the training model:
* Monitors key metrics (e.g. loss, accuracy, and learning rates)
* Adjusts in real-time and improves overall model performance
* Showcases possible overfitting or underfitting cases

Note: <font color='#FFC067'>Frequently refresh</font> the button in the top right of the TensorBoard.

In [ ]:
%load_ext tensorboard
%tensorboard --logdir /content/runs/detect/train

## 🎯 Training the YOLOv11 Model
Key Training Parameters:
* `imgsz` defines the <font color='#FFC067'>target image size</font> of the inputs for consistency
* `batch` processes the <font color='#FFC067'>number of images</font> concurrently  
* `epochs` establish the <font color='#FFC067'>number of complete cycles</font> for training

Note: It is important to check the accuracy, find class weaknesses, and track improvements within the model.
  

In [ ]:
# Load YOLOv11 model
model = YOLO("yolo11n.pt")

# Set dataset configuration to YAML file
dataset_config = os.path.join(data_dir, 'data.yaml') # Load all class labels

# Train the model
results = model.train(
    data=dataset_config,
    epochs=100,
    batch=64,  # Set appropriate batch size
    imgsz=640,  # Standardize image size for training
    plots=True,
    patience=50
)

# Inspect training results
print(results)

# Augmenting Data

In [ ]:
import albumentations as A
from albumentations.pytorch import ToTensorV2
import cv2
import os

# Define augmentation pipeline
transform = A.Compose([
    A.HorizontalFlip(p=0.5),
    A.Rotate(limit=45, p=0.7),
    A.RandomBrightnessContrast(p=0.3),
    A.ShiftScaleRotate(shift_limit=0.1, scale_limit=0.2, rotate_limit=25, p=0.5),
    A.RandomCrop(width=128, height=128),
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ToTensorV2(),
])

# Batch process images in a folder
input_dir = '/path/to/train/images'
output_dir = '/path/to/augmented/images'

def augment_images(input_dir, output_dir, transform):
    if not os.path.exists(output_dir):
        os.makedirs(output_dir)
    
    for image_file in os.listdir(input_dir):
        image_path = os.path.join(input_dir, image_file)
        image = cv2.imread(image_path)
        
        # Apply augmentation
        augmented = transform(image=image)['image']
        
        # Save augmented image
        output_path = os.path.join(output_dir, image_file)
        cv2.imwrite(output_path, augmented.numpy().transpose(1, 2, 0) * 255)  # Convert back to original range

augment_images(input_dir, output_dir, transform)

## 📊 Evaluating the Model
* Use validation data and metrics to evaluate the model performance

In [ ]:
from IPython.display import Image, display
import os

# Set the base directory
base_dir = "/content/runs/detect/train/"

# List of filenames to display
filenames = [
    "labels.jpg",
    "F1_curve.png",
    "PR_curve.png",
    "P_curve.png",
    "R_curve.png",
    "confusion_matrix.png",
    "confusion_matrix_normalized.png"
]

# Display each image
for filename in filenames:
    image_path = os.path.join(base_dir, filename)
    display(Image(image_path))

## 🔎 Running an Inference
* <font color = "67F2FF">Test real-world application</font> for the model
  
Configure the two inference parameters as needed: `conf` and `iou`
  * <font color = "67F2FF">Confidence Threshold</font> (conf):
    * Sets minimum probability score required for <font color = "67F2FF">keeping detections</font>  and predicted bounding box
    * Adjust appropriately to reduce either low-confidence guesses or false positives
  * <font color = "67F2FF">Intersection over Union Threshold</font> (iou):
    * Sets Non-Maximum Suppression (NMS) value required for <font color = "67F2FF">removing duplicates</font> and overlapping bounding boxes
    * Adjust appropriately to control the strength of merging and deleting neighboring duplicate detections

In [ ]:
# Download the video to test the model

!wget https://huggingface.co/datasets/OceanCV/PlasticTank_Video/resolve/main/tankvid.mp4?download=true -O tankvid.mp4

In [ ]:
import cv2
from ultralytics import YOLO

model_path = '/content/runs/detect/train/weights/best.pt'
model = YOLO(model_path)

video_path = 'tankvid.mp4'

results = model.track(source=video_path, save=True, show=True, conf=0.25, iou=0.7)